In [ ]:
# custom imports
import nekt
import os                           
from dotenv import load_dotenv      
from typing import Optional 
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# get Nekt data access token        
load_dotenv()
nekt.data_access_token = os.getenv("NEKT_DATA_ACCESS_TOKEN")

# auxiliary functions
def load_nekt_table(table_name: str, layer_name: str) -> DataFrame:
    return nekt.load_table(layer_name=layer_name, table_name=table_name)

def save_nekt_table(
    df: DataFrame, 
    layer_name: str, 
    table_name: str,
    folder_name: Optional[str] = None 
):
    nekt.save_table(
        df=df,
        layer_name=layer_name,
        table_name=table_name,
        folder_name=folder_name
    )

def display(df: DataFrame, limit=10):
    return df.limit(limit).toPandas()

# process:
##  load source tables
### clickup tables
df_bronze_clickup_users        = load_nekt_table("users"        , "Bronze")
df_bronze_clickup_spaces       = load_nekt_table("spaces"       , "Bronze")
df_bronze_clickup_time_entries = load_nekt_table("time_entries" , "Bronze")

### contaazul tables
df_bronze_contaazul_account_payables    = load_nekt_table("despesas"        , "Bronze")
df_bronze_contaazul_account_receivables = load_nekt_table("receitas"        , "Bronze")
df_bronze_contaazul_installments        = load_nekt_table("parcelas"        , "Bronze")
df_bronze_contaazul_sales               = load_nekt_table("time_entries"    , "Bronze")
# df_bronze_contaazul_clientes = load_nekt_table("time_entries" , "Bronze")
# df_bronze_contaazul_vendas = load_nekt_table("time_entries" , "Bronze")
# df_bronze_contaazul_pessoas = load_nekt_table("time_entries" , "Bronze")

## filter columns
### clickup filters
# df_silver_clickup_users = df_bronze_clickup_users.select(
#       F.col("id").cast("integer")     .alias("user_id")
#     , F.col("username")               .alias("user_name")
# ).filter(
#     F.col("user_name").isNotNull()
# ).dropDuplicates(
#     ["user_id"]
# )

# df_silver_clickup_spaces = df_bronze_clickup_spaces.select(
#       F.col("id").cast("long")        .alias("space_id")
#     , F.col("name")                   .alias("space_name")
# ).filter(
#     F.col("space_name").isNotNull()
# ).dropDuplicates(
#     ["space_id"]
# )

# df_silver_clickup_time_entries = df_bronze_clickup_time_entries.select(
#       F.col("id").cast("long")                        .alias("interval_id")
#     , F.col("wid").cast("integer")                    .alias("team_id")
#     , F.col("user.id").cast("integer")                .alias("user_id")
#     , F.col("user.username")                          .alias("user_name")
#     , F.col("task_location.space_id").cast("long")    .alias("space_id")
#     , F.col("task.id").alias("task_id")
#     , F.col("start").cast("long")                     .alias("interval_date_start_ms")
#     , F.to_timestamp(F.col("start") / 1000)           .alias("interval_date_start_iso")
#     , F.col("end").cast("long")                       .alias("interval_date_end_ms")
#     , F.to_timestamp(F.col("end") / 1000)             .alias("interval_date_end_iso")
#     , F.col("at").cast("long")                        .alias("interval_date_added_ms")
#     , F.to_timestamp(F.col("at") / 1000)              .alias("interval_date_added_iso")
# ).filter(
#       F.col("interval_id").isNotNull()
#     & F.col("user_id").isNotNull()
#     & F.col("interval_date_start_ms").isNotNull()
#     & F.col("interval_date_end_ms").isNotNull()
#     & (F.col("interval_date_end_ms") > F.col("interval_date_start_ms"))  # validating business rule
# ).dropDuplicates(
#     ["interval_id"]
# )

### contaazul filters
# df_bronze_contaazul_account_payables = df_bronze_contaazul_account_payables.select(
#       F.col("id")
#     , F.col("descricao")
#     , F.col("data_vencimento")
#     , F.col("status")
#     , F.col("total")
#     , F.col("nao_pago")
#     , F.col("pago")
#     , F.col("data_criacao")
#     , F.col("data_alteracao")
#     # , F.col("categoria_principal_id")
#     # , F.col("categoria_principal_nome")
#     # , F.col("fornecedor_id")
#     # , F.col("fornecedor_nome")
#     # , F.col("_loaded_at")
# )

# df_bronze_contaazul_account_receivables = df_bronze_contaazul_account_receivables.select(
#       F.col("id")
#     , F.col("descricao")
#     , F.col("data_vencimento")
#     , F.col("status")
#     , F.col("total")
#     , F.col("nao_pago")
#     , F.col("pago")
#     , F.col("data_criacao")
#     , F.col("data_alteracao")
#     # , F.col("categoria_principal_id")
#     # , F.col("categoria_principal_nome")
#     # , F.col("fornecedor_id")
#     # , F.col("fornecedor_nome")
#     # , F.current_timestamp().alias("loaded_at")
# )

df_bronze_contaazul_installments = df_bronze_contaazul_installments.select(
      F.col("id")                       .alias("parcela_id")
    , F.col("status")                   .alias("parcela_status")
    , F.col("evento.condicao_pagamento").alias("condicao_pagamento")
    , F.col("referencia")
    , F.col("evento.agendado")          .alias("agendado")
    , F.col("evento.tipo")              .alias("tipo_evento")
    , F.col("")                         .alias("")
)

# 
df_bronze_contaazul_settled_installments = df_bronze_contaazul_installments.select(
      F.col("id")                                       .alias("parcela_id")
    , F.explode("baixas")                               .alias("baixa")).filter(
        F.size("baixas") > 0 
    ).select(
          F.col("parcela_id")
        , F.col("baixa.id")                             .alias("baixa_id")
        , F.col("baixa.versao")                         .alias("baixa_versao")
        , F.col("baixa.data_pagamento")                 .alias("baixa_data_pagamento") 
        , F.col("baixa.id_reconciliacao")               .alias("baixa_id_reconciliacao")
        , F.col("baixa.id_solicitacao_cobranca")        .alias("baixa_solicitacao_cobranca")
        , F.col("baixa.observacao")                     .alias("baixa_observacao")
        , F.col("baixa.metodo_pagamento")               .alias("baixa_metodo_pagamento")
        , F.col("baixa.origem")                         .alias("baixa_origem")
        , F.col("baixa.id_recibo_digital")              .alias("baixa_id_recibo_digital")
        , F.col("baixa.tipo_evento_financeiro")         .alias("baixa_tipo_evento_financeiro")
        , F.col("baixa.nsu")                            .alias("baixa_nsu")
        , F.col("baixa.id_referencia")                  .alias("baixa_id_referencia")
        , F.col("baixa.atualizado_em")                  .alias("baixa_atualizado_em")
        , F.col("baixa.valor_composicao.desconto")      .alias("baixa_desconto")
        , F.col("baixa.valor_composicao.juros")         .alias("baixa_juros")
        , F.col("baixa.valor_composicao.multa")         .alias("baixa_multa")
        , F.col("baixa.valor_composicao.taxa")          .alias("baixa_taxa")
        , F.col("baixa.valor_composicao.valor_bruto")   .alias("baixa_valor_bruto")
        , F.col("baixa.valor_composicao.valor_liquido") .alias("baixa_valor_liquido")
        , F.current_timestamp()                         .alias("loaded_at")
    )

## save results to output tables
# save_nekt_table(df_silver_users         , "Silver", "users",        folder_name="studio_61_clickup")
# save_nekt_table(df_silver_spaces        , "Silver", "spaces",       folder_name="studio_61_clickup")
# save_nekt_table(df_silver_time_entries  , "Silver", "time_entries", folder_name="studio_61_clickup")

params: {'include_layer_database_name': 'true'}
Table details: {'id': '7adcd28a-c0e3-4a6a-aea2-1eeef140cee7', 'name': 'users', 'slug': 'users', 'database_id': 'users', 'layer_database_name': 'stech_solucoes_tecnologicas_bronze', 'folder': '2a5ad9ed-e5db-46ed-b2f8-3bf559fae219', 'description': None, 's3_path': 'bq://nekt-hosted-infrastructure.stech_solucoes_tecnologicas_bronze.users', 'layer': '85675f10-b2e9-44ea-8e30-a88b0b545b89', 'deleted': False, 'created_at': '2026-02-26T11:33:03.805939-03:00', 'updated_at': '2026-03-02T15:28:42.292806-03:00'}
params: {'include_layer_database_name': 'true'}
Table details: {'id': '5169a424-7ea2-43be-98e5-f26828090499', 'name': 'spaces', 'slug': 'spaces', 'database_id': 'spaces', 'layer_database_name': 'stech_solucoes_tecnologicas_bronze', 'folder': '2a5ad9ed-e5db-46ed-b2f8-3bf559fae219', 'description': None, 's3_path': 'bq://nekt-hosted-infrastructure.stech_solucoes_tecnologicas_bronze.spaces', 'layer': '85675f10-b2e9-44ea-8e30-a88b0b545b89', 'delet

,parcela_id,baixa_id,baixa_versao,baixa_data_pagamento,baixa_id_reconciliacao,baixa_solicitacao_cobranca,baixa_observacao,baixa_metodo_pagamento,baixa_origem,baixa_id_recibo_digital,...,baixa_nsu,baixa_id_referencia,baixa_atualizado_em,baixa_desconto,baixa_juros,baixa_multa,baixa_taxa,baixa_valor_bruto,baixa_valor_liquido,loaded_at
0,6b9c1e28-deb1-4e1a-bd2d-51e3dc4ed6b9,4df9a58c-655e-43da-918f-1218395e7469,0,2026-01-12,347e4703-cb00-4ab7-9dcf-3c33acd20312,None,None,OUTRO,LANCAMENTO_FINANCEIRO,None,...,None,None,2026-01-13T14:18:55,0.0,0.0,0.0,0.0,1000.00,1000.00,2026-03-16 17:00:06.458866
1,516673fa-8698-4543-b190-62f24db8bae7,fe6b1a0d-5817-4630-b7a7-ac27bad0592f,0,2026-01-26,52073256-af27-4349-b311-72352b83266c,None,None,OUTRO,LANCAMENTO_FINANCEIRO,None,...,None,None,2026-01-27T10:35:29,0.0,0.0,0.0,0.0,1800.00,1800.00,2026-03-16 17:00:06.458866
2,508c236e-0b85-43b8-b2ed-24ab6e2df8fa,6bb77e68-ba4b-4724-a6ee-7a8d77af207f,1,2026-02-25,037b5254-c182-47ba-a1f3-cd2ee0fcd2a9,None,None,OUTRO,LANCAMENTO_FINANCEIRO,None,...,None,None,2026-02-27T18:27:39,0.0,0.0,0.0,0.0,3060.00,3060.00,2026-03-16 17:00:06.458866
3,a128e821-b056-41dd-b3d8-a65ed869c623,e6adc392-2e7f-402b-86e1-735914d7e862,1,2026-02-25,037b5254-c182-47ba-a1f3-cd2ee0fcd2a9,None,None,CARTAO_CREDITO,LANCAMENTO_FINANCEIRO,None,...,None,None,2026-02-27T18:27:39,0.0,0.0,0.0,0.0,350.80,350.80,2026-03-16 17:00:06.458866
4,122db294-01fc-4ab9-8db0-d01f5ee0667d,323bdc0e-24a3-4b82-9d94-12f387d2a2b3,0,2026-01-05,945ae432-0cb5-4897-ab72-78e0a8fed667,None,None,OUTRO,LANCAMENTO_FINANCEIRO,None,...,None,None,2026-01-06T10:33:20,0.0,0.0,0.0,0.0,2597.00,2597.00,2026-03-16 17:00:06.458866
5,d4a14378-dc7c-4d14-8bc2-13a10b1e0559,a5ceb242-4e85-4fa8-82db-703a7fc4e88b,1,2026-02-05,4679e4c1-f4c8-43aa-b160-4d99ed00c925,None,None,OUTRO,LANCAMENTO_FINANCEIRO,None,...,None,None,2026-02-06T09:19:40,0.0,0.0,0.0,0.0,2597.00,2597.00,2026-03-16 17:00:06.458866
6,0a0324a9-b2ab-478a-a588-375157228777,310e7682-5234-4aa1-8bcf-94dfc1fc236d,0,2026-03-05,5f1d3e2a-c877-4d4c-af31-cfe9e9b53cb0,None,None,OUTRO,LANCAMENTO_FINANCEIRO,None,...,None,None,2026-03-06T08:27:18,0.0,0.0,0.0,0.0,2597.00,2597.00,2026-03-16 17:00:06.458866
7,fb347dae-87b5-4c08-80d6-a4ab85f64504,94f8b762-c190-45fa-a10a-828c6372f304,1,2026-02-04,21c1698e-6534-4683-9734-baa3da9b1fb0,None,None,OUTRO,LANCAMENTO_FINANCEIRO,None,...,None,None,2026-02-05T09:07:42,0.0,0.0,0.0,0.0,30.00,30.00,2026-03-16 17:00:06.458866
8,23771bfe-b5fe-4774-a930-959292ded2e7,c1cd08d7-3444-4cd4-892c-817bdc2afdf4,1,2026-01-21,b704b788-481e-45f9-b6c4-ae3a3dcc28d2,None,None,OUTRO,LANCAMENTO_FINANCEIRO,None,...,None,None,2026-01-22T09:33:01,0.0,0.0,0.0,0.0,30.00,30.00,2026-03-16 17:00:06.458866
9,88fdfcb8-31a9-42bd-a766-b9a4e20b3aeb,b35a3ce9-08e5-41ad-8db6-99413bf7ea35,0,2026-01-26,52073256-af27-4349-b311-72352b83266c,None,None,OUTRO,LANCAMENTO_FINANCEIRO,None,...,None,None,2026-01-27T10:35:29,0.0,0.0,0.0,0.0,718.89,718.89,2026-03-16 17:00:06.458866
